# 🤖 Notebook 3: Machine Learning Models
**Preprocessing → Feature Engineering → Training → Tuning → Evaluation**

This is the core ML notebook. We build, compare, and tune three classifiers to predict smoking status.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')
os.makedirs('plots', exist_ok=True)

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              confusion_matrix, classification_report, roc_auc_score, roc_curve)
from sklearn.inspection import permutation_importance

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
print("✅ All imports successful")


## 1. Load & Preprocess Data

> **Why preprocess?** Raw data has different scales, outliers, and possibly noisy features. ML models work best when data is clean and normalized.

In [ ]:
# Load data
df = pd.read_excel('test11.xlsx')
df.drop(columns=['id'], inplace=True, errors='ignore')
df.drop_duplicates(inplace=True)

print(f"Shape: {df.shape}")
print(f"Target distribution:\n{df['smoking'].value_counts()}")


In [ ]:
# ── Step 1: Separate features and target ─────────────────────────────────────
X = df.drop(columns=['smoking'])
y = df['smoking']

print(f"Features: {X.shape[1]} | Samples: {X.shape[0]}")
print(f"Feature names: {X.columns.tolist()}")


In [ ]:
# ── Step 2: Outlier Capping (IQR method) ─────────────────────────────────────
# We cap outliers instead of removing them — this preserves data size.
# Highly skewed features get capped at Q1-1.5*IQR and Q3+1.5*IQR.

def cap_outliers_iqr(df, cols):
    df = df.copy()
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower, upper = Q1 - 1.5*IQR, Q3 + 1.5*IQR
        df[col] = df[col].clip(lower, upper)
    return df

# Identify highly skewed columns
skewed_cols = X.skew()[X.skew().abs() > 1].index.tolist()
print(f"Capping outliers in {len(skewed_cols)} skewed columns: {skewed_cols}")
X = cap_outliers_iqr(X, skewed_cols)
print("✅ Outlier capping done")


In [ ]:
# ── Step 3: Feature Engineering ──────────────────────────────────────────────
# Add BMI — a clinically meaningful combined feature
X['BMI'] = X['weight(kg)'] / (X['height(cm)'] / 100) ** 2

# Blood pressure ratio (pulse pressure) — cardiovascular indicator
X['pulse_pressure'] = X['systolic'] - X['relaxation']

# Liver stress score — both liver enzymes elevated together
X['liver_stress'] = X['AST'] + X['ALT'] + X['Gtp']

print("✅ New features added:", ['BMI', 'pulse_pressure', 'liver_stress'])
print(f"Feature count: {X.shape[1]}")


In [ ]:
# ── Step 4: Train / Validation / Test Split (60/20/20) ───────────────────────
# We use 3 splits:
# - Train: model learns from this
# - Validation: used for hyperparameter tuning
# - Test: final unbiased evaluation

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=RANDOM_STATE, stratify=y_temp)
# 0.25 of 80% = 20% of total

print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_val.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")
print(f"\nClass balance in train:\n{y_train.value_counts()}")


In [ ]:
# ── Step 5: Scaling ───────────────────────────────────────────────────────────
# StandardScaler: zero mean, unit variance — best for most models
# We fit ONLY on train set, then transform train/val/test separately.
# (Fitting on test would 'leak' information — a common mistake!)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # Fit + transform
X_val_scaled = scaler.transform(X_val)            # Only transform
X_test_scaled = scaler.transform(X_test)          # Only transform

print("✅ StandardScaler applied")
print(f"Train mean (should be ≈0): {X_train_scaled.mean(axis=0).mean():.4f}")
print(f"Train std  (should be ≈1): {X_train_scaled.std(axis=0).mean():.4f}")


## 2. Model Training

> We train 3 models:
> 1. **Logistic Regression** — Simple, interpretable, great baseline
> 2. **Naive Bayes** — Fast, probabilistic, works well with independent features
> 3. **MLP Neural Network** — More complex, can capture non-linear patterns

All models use the same scaled data for fair comparison.

In [ ]:
# ── Helper: evaluate a model ─────────────────────────────────────────────────
def evaluate_model(name, model, X_tr, y_tr, X_ev, y_ev):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_ev)
    y_prob = model.predict_proba(X_ev)[:, 1] if hasattr(model, 'predict_proba') else None
    
    results = {
        'Model': name,
        'Accuracy': accuracy_score(y_ev, y_pred),
        'Precision': precision_score(y_ev, y_pred, zero_division=0),
        'Recall': recall_score(y_ev, y_pred, zero_division=0),
        'F1': f1_score(y_ev, y_pred, zero_division=0),
        'ROC-AUC': roc_auc_score(y_ev, y_prob) if y_prob is not None else None
    }
    return results, y_pred, model


In [ ]:
# ── Model 1: Logistic Regression ─────────────────────────────────────────────
# max_iter=1000 ensures convergence on this dataset
lr_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, C=1.0)
lr_results, lr_pred, lr_model = evaluate_model(
    'Logistic Regression', lr_model, X_train_scaled, y_train, X_val_scaled, y_val)

print("📌 Logistic Regression — Validation Results:")
for k, v in lr_results.items():
    if k != 'Model' and v is not None:
        print(f"  {k}: {v:.4f}")


In [ ]:
# ── Feature importance from Logistic Regression coefficients ─────────────────
# Larger |coefficient| = more influence on prediction
coefs = pd.Series(lr_model.coef_[0], index=X_train.columns).sort_values(key=abs, ascending=False)

plt.figure(figsize=(10, 5))
colors = ['tomato' if v > 0 else 'steelblue' for v in coefs.values]
coefs.plot(kind='bar', color=colors, edgecolor='white')
plt.axhline(0, color='black', linewidth=0.8)
plt.title('Logistic Regression — Feature Coefficients (Feature Importance)', fontweight='bold')
plt.ylabel('Coefficient Value')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('plots/lr_coefficients.png', bbox_inches='tight')
plt.show()
print("\nTop 5 features (positive = increases smoking probability):")
print(coefs.head(5).round(4))


In [ ]:
# ── Model 2: Naive Bayes ─────────────────────────────────────────────────────
# GaussianNB assumes each feature follows a normal distribution per class.
# It calculates P(smoking | features) using Bayes theorem.
nb_model = GaussianNB()
nb_results, nb_pred, nb_model = evaluate_model(
    'Naive Bayes', nb_model, X_train_scaled, y_train, X_val_scaled, y_val)

print("📌 Naive Bayes — Validation Results:")
for k, v in nb_results.items():
    if k != 'Model' and v is not None:
        print(f"  {k}: {v:.4f}")

print("\nClass probability interpretation example (first 5 validation samples):")
probs = nb_model.predict_proba(X_val_scaled[:5])
for i, (p0, p1) in enumerate(probs):
    print(f"  Sample {i+1}: P(non-smoker)={p0:.3f}, P(smoker)={p1:.3f} → Predicted: {'Smoker' if p1>0.5 else 'Non-smoker'}")


In [ ]:
# ── Model 3: MLP Neural Network ──────────────────────────────────────────────
# Architecture: Input → 64 → 32 → Output
# ReLU activation: passes positive values, kills negatives → handles non-linearity
# adam optimizer: adaptive learning rate (works well for most problems)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),     # Two hidden layers
    activation='relu',               # ReLU activation
    solver='adam',                   # Adaptive optimizer
    max_iter=500,                    # More iterations for convergence
    early_stopping=True,             # Stop if validation loss stops improving
    validation_fraction=0.1,         # 10% of train as internal val
    random_state=RANDOM_STATE,
    alpha=0.01                       # L2 regularization to prevent overfitting
)
mlp_results, mlp_pred, mlp_model = evaluate_model(
    'MLP Neural Network', mlp_model, X_train_scaled, y_train, X_val_scaled, y_val)

print("📌 MLP Neural Network — Validation Results:")
for k, v in mlp_results.items():
    if k != 'Model' and v is not None:
        print(f"  {k}: {v:.4f}")

print(f"\nArchitecture: Input({X_train_scaled.shape[1]}) → 64 → 32 → Output(2)")
print(f"Iterations completed: {mlp_model.n_iter_}")


## 3. Confusion Matrices

> A confusion matrix shows how many predictions were correct and where the model made mistakes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
models_info = [
    ('Logistic Regression', lr_pred, 'Blues'),
    ('Naive Bayes', nb_pred, 'Greens'),
    ('MLP Neural Network', mlp_pred, 'Oranges'),
]

for ax, (name, pred, cmap) in zip(axes, models_info):
    cm = confusion_matrix(y_val, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap, ax=ax,
                xticklabels=['Non-smoker', 'Smoker'],
                yticklabels=['Non-smoker', 'Smoker'])
    acc = accuracy_score(y_val, pred)
    ax.set_title(f'{name}\nAccuracy: {acc:.3f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.suptitle('Confusion Matrices — Validation Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/confusion_matrices.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Classification Reports ────────────────────────────────────────────────────
for name, pred in [('Logistic Regression', lr_pred), ('Naive Bayes', nb_pred), ('MLP', mlp_pred)]:
    print(f"\n{'='*50}")
    print(f"Classification Report: {name}")
    print('='*50)
    print(classification_report(y_val, pred, target_names=['Non-smoker', 'Smoker']))


## 4. Model Comparison

In [ ]:
# ── Comparison Table ──────────────────────────────────────────────────────────
comparison_df = pd.DataFrame([lr_results, nb_results, mlp_results])
comparison_df = comparison_df.set_index('Model')
comparison_df = comparison_df.round(4)
print("\n📊 MODEL COMPARISON TABLE (Validation Set):")
print(comparison_df.to_string())


In [ ]:
# ── Visual comparison ─────────────────────────────────────────────────────────
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(metrics))
width = 0.25
bars_lr = ax.bar(x - width, comparison_df.loc['Logistic Regression', metrics], width,
                  label='Logistic Regression', color='steelblue', edgecolor='white')
bars_nb = ax.bar(x, comparison_df.loc['Naive Bayes', metrics], width,
                  label='Naive Bayes', color='seagreen', edgecolor='white')
bars_mlp = ax.bar(x + width, comparison_df.loc['MLP Neural Network', metrics], width,
                   label='MLP Neural Network', color='tomato', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — All Metrics (Validation Set)', fontweight='bold')
ax.legend()
ax.axhline(0.5, color='gray', linestyle='--', alpha=0.4)

# Value labels on bars
for bars in [bars_lr, bars_nb, bars_mlp]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01, f'{h:.2f}',
                ha='center', va='bottom', fontsize=7)

plt.tight_layout()
plt.savefig('plots/model_comparison.png', bbox_inches='tight')
plt.show()


## 5. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
model_data = [
    ('Logistic Regression', lr_model, 'steelblue'),
    ('Naive Bayes', nb_model, 'seagreen'),
    ('MLP Neural Network', mlp_model, 'tomato'),
]

for name, model, color in model_data:
    probs = model.predict_proba(X_val_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, probs)
    auc = roc_auc_score(y_val, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1], [0,1], 'k--', alpha=0.4, label='Random (AUC=0.50)')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — All Models', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plots/roc_curves.png', bbox_inches='tight')
plt.show()


## 6. Hyperparameter Tuning with GridSearchCV

> **What are hyperparameters?** Settings you choose before training that control how a model learns (e.g., regularization strength, number of neurons). They're not learned from data — you have to tune them.

> **GridSearchCV** tries every combination you specify and uses cross-validation to pick the best one. It's the gold standard for tuning.

In [ ]:
# ── Tune Logistic Regression ─────────────────────────────────────────────────
# C: regularization strength (smaller C = more regularization = simpler model)
# penalty: type of regularization (l1=sparse, l2=smooth)
# solver: algorithm for optimization

param_grid_lr = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l2'],
    'solver': ['lbfgs', 'liblinear'],
    'max_iter': [1000]
}

grid_lr = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    param_grid=param_grid_lr,
    cv=5,           # 5-fold cross-validation
    scoring='f1',   # Optimize for F1 (balanced metric)
    n_jobs=-1,      # Use all CPU cores
    verbose=0
)

grid_lr.fit(X_train_scaled, y_train)

print(f"✅ Grid Search Complete!")
print(f"Best parameters: {grid_lr.best_params_}")
print(f"Best cross-validation F1: {grid_lr.best_score_:.4f}")


In [ ]:
# ── Tune MLP ─────────────────────────────────────────────────────────────────
param_grid_mlp = {
    'hidden_layer_sizes': [(64, 32), (128, 64), (32, 16)],
    'alpha': [0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01],
}

grid_mlp = GridSearchCV(
    MLPClassifier(max_iter=500, early_stopping=True, random_state=RANDOM_STATE),
    param_grid=param_grid_mlp,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=0
)

grid_mlp.fit(X_train_scaled, y_train)

print(f"✅ MLP Grid Search Complete!")
print(f"Best parameters: {grid_mlp.best_params_}")
print(f"Best cross-validation F1: {grid_mlp.best_score_:.4f}")


## 7. Final Model Evaluation on Test Set

> **Important:** We only use the test set ONCE — at the very end. Using it earlier would give us overly optimistic results.

In [ ]:
# ── Evaluate all 4 models on test set ────────────────────────────────────────
# (original + tuned versions)

def test_eval(name, model, X_te, y_te):
    pred = model.predict(X_te)
    prob = model.predict_proba(X_te)[:, 1]
    return {
        'Model': name,
        'Accuracy': round(accuracy_score(y_te, pred), 4),
        'Precision': round(precision_score(y_te, pred, zero_division=0), 4),
        'Recall': round(recall_score(y_te, pred, zero_division=0), 4),
        'F1': round(f1_score(y_te, pred, zero_division=0), 4),
        'ROC-AUC': round(roc_auc_score(y_te, prob), 4),
    }

final_results = [
    test_eval('Logistic Regression (default)', lr_model, X_test_scaled, y_test),
    test_eval('Naive Bayes', nb_model, X_test_scaled, y_test),
    test_eval('MLP Neural Network (default)', mlp_model, X_test_scaled, y_test),
    test_eval('Logistic Regression (tuned)', grid_lr.best_estimator_, X_test_scaled, y_test),
    test_eval('MLP Neural Network (tuned)', grid_mlp.best_estimator_, X_test_scaled, y_test),
]

final_df = pd.DataFrame(final_results).set_index('Model')
print("\n🏆 FINAL MODEL COMPARISON — TEST SET")
print("="*70)
print(final_df.to_string())
print("\n✅ Best model:", final_df['F1'].idxmax())
print(f"   F1 Score: {final_df['F1'].max():.4f}")


## 8. Final Production Pipeline

> **A clean, end-to-end pipeline** that can be used in production: load new data, preprocess, and get predictions.

In [ ]:
from sklearn.pipeline import Pipeline

# ── Determine best model ──────────────────────────────────────────────────────
best_model_name = final_df['F1'].idxmax()
if 'MLP' in best_model_name and 'tuned' in best_model_name:
    best_estimator = grid_mlp.best_estimator_
elif 'Logistic' in best_model_name and 'tuned' in best_model_name:
    best_estimator = grid_lr.best_estimator_
elif 'MLP' in best_model_name:
    best_estimator = mlp_model
elif 'Naive' in best_model_name:
    best_estimator = nb_model
else:
    best_estimator = lr_model

print(f"Selected best model: {best_model_name}")

# ── Final pipeline ────────────────────────────────────────────────────────────
final_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', best_estimator)
])

# Re-fit on ALL available data (train + val) for maximum generalization
X_all_trainval = pd.concat([X_train, X_val])
y_all_trainval = pd.concat([y_train, y_val])
final_pipeline.fit(X_all_trainval, y_all_trainval)

# Final test evaluation
y_final_pred = final_pipeline.predict(X_test)
y_final_prob = final_pipeline.predict_proba(X_test)[:, 1]

print(f"\n🎯 FINAL PIPELINE — Test Set Performance")
print(f"  Accuracy:  {accuracy_score(y_test, y_final_pred):.4f}")
print(f"  Precision: {precision_score(y_test, y_final_pred):.4f}")
print(f"  Recall:    {recall_score(y_test, y_final_pred):.4f}")
print(f"  F1 Score:  {f1_score(y_test, y_final_pred):.4f}")
print(f"  ROC-AUC:   {roc_auc_score(y_test, y_final_prob):.4f}")


In [ ]:
# ── Example prediction on new data ───────────────────────────────────────────
print("\n🔮 Example: Predict smoking status for a new patient")
print("="*55)

# Create a sample patient record (with feature-engineered columns)
sample_patient = X_test.iloc[[0]].copy()
true_label = y_test.iloc[0]

prediction = final_pipeline.predict(sample_patient)[0]
probability = final_pipeline.predict_proba(sample_patient)[0]

print(f"Patient features (selected):")
print(f"  hemoglobin: {sample_patient['hemoglobin'].values[0]:.1f}")
print(f"  Gtp:        {sample_patient['Gtp'].values[0]:.1f}")
print(f"  BMI:        {sample_patient['BMI'].values[0]:.1f}")
print(f"\nPrediction:         {'🚬 Smoker' if prediction == 1 else '✅ Non-smoker'}")
print(f"Confidence:         {max(probability)*100:.1f}%")
print(f"Actual label:       {'🚬 Smoker' if true_label == 1 else '✅ Non-smoker'}")
print(f"Correct:            {'✅ Yes' if prediction == true_label else '❌ No'}")


## 9. Final Summary & Recommendations

### What We Built
A complete ML pipeline with:
- **Data cleaning:** duplicate removal, outlier capping
- **Feature engineering:** BMI, pulse pressure, liver stress score
- **Train/Val/Test split** to prevent data leakage
- **StandardScaler** for feature normalization
- **3 models trained:** Logistic Regression, Naive Bayes, MLP Neural Network
- **GridSearchCV** for hyperparameter tuning
- **Full evaluation:** accuracy, precision, recall, F1, ROC-AUC

### Why The Best Model Won
- Logistic Regression is strong here because the features have **linear-ish relationships** with smoking
- MLP can capture **non-linear patterns** but needs more data to truly shine
- Naive Bayes is fast but suffers when features are **correlated** (which some are here)

### For Viva Discussion
1. **Why StandardScaler?** Logistic Regression and MLP are distance/gradient based — different scales distort the learning
2. **Why cap outliers instead of remove?** Preserves dataset size; critical with 217 rows
3. **Why 60/20/20 split?** Standard split ensuring enough data in each partition
4. **What is GridSearchCV?** K-fold CV on all hyperparameter combinations; picks the one with best cross-validated score
5. **Why F1 over accuracy?** F1 balances precision and recall; more robust for clinical predictions